In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
Implement decode-phase multi-head attention where the key and value caches are stored as
<code>int8</code> with per-token scale factors. This memory layout halves KV-cache bandwidth
versus <code>float32</code> and is used in production LLM serving systems such as TensorRT-LLM
and vLLM. Given a query tensor <code>Q</code> for a single new token, <code>int8</code> key cache
<code>K_int8</code>, <code>int8</code> value cache <code>V_int8</code>, and per-token scales
<code>k_scale</code> and <code>v_scale</code>, dequantize the caches and compute scaled
dot-product attention to produce <code>output</code>. All non-integer tensors use
<code>float32</code>.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Implement the function <code>solve(Q, K_int8, V_int8, k_scale, v_scale, output, num_heads, seq_len, head_dim)</code>.</li>
  <li>Do not change the function signature or use external libraries beyond the standard GPU frameworks.</li>
  <li>Write the result into the provided <code>output</code> buffer.</li>
  <li>Dequantize using per-token scales: <code>K_float[h, s, d] = K_int8[h, s, d] &times; k_scale[h, s]</code> (and analogously for V).</li>
  <li>Use scaled dot-product attention with scale factor <code>1 / sqrt(head_dim)</code> and a softmax over the sequence dimension.</li>
</ul>

<h2>Example</h2>
<p>
  With <code>num_heads</code> = 1, <code>seq_len</code> = 3, <code>head_dim</code> = 4:
</p>
<p>
  <strong>Input:</strong><br>
  $Q$ (1&times;4):
  $$
  \begin{bmatrix} 1 & 1 & 1 & 1 \end{bmatrix}
  $$
  $K\_int8$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 10 & 0 & 0 & 0 \\ 0 & 10 & 0 & 0 \\ 0 & 0 & 10 & 0 \end{bmatrix}
  $$
  $k\_scale$ (1&times;3): $\begin{bmatrix} 0.1 & 0.1 & 0.1 \end{bmatrix}$
  &nbsp;&rArr;&nbsp;
  $K\_float$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{bmatrix}
  $$
  $V\_int8$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 10 & 20 & 30 & 40 \\ 50 & 60 & 70 & 80 \\ 90 & 100 & 110 & 120 \end{bmatrix}
  $$
  $v\_scale$ (1&times;3): $\begin{bmatrix} 0.1 & 0.1 & 0.1 \end{bmatrix}$
  &nbsp;&rArr;&nbsp;
  $V\_float$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{bmatrix}
  $$
</p>
<p>
  Scores = $Q \cdot K\_float^T / \sqrt{4}$ = $\begin{bmatrix} 0.5 & 0.5 & 0.5 \end{bmatrix}$,
  so <em>softmax</em> weights = $\begin{bmatrix} 1/3 & 1/3 & 1/3 \end{bmatrix}$.
</p>
<p>
  <strong>Output</strong> (1&times;4):
  $$
  \begin{bmatrix} 5.00 & 6.00 & 7.00 & 8.00 \end{bmatrix}
  $$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>num_heads</code> &le; 64</li>
  <li>1 &le; <code>seq_len</code> &le; 32,768</li>
  <li>8 &le; <code>head_dim</code> &le; 256; <code>head_dim</code> is a multiple of 8</li>
  <li><code>K_int8</code> and <code>V_int8</code> values are in $[-128, 127]$</li>
  <li>All scale values are positive <code>float32</code></li>
  <li>Performance is measured with <code>num_heads</code> = 32, <code>seq_len</code> = 8,192, <code>head_dim</code> = 128</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K_int8, V_int8, k_scale, v_scale, output are device pointers
extern "C" void solve(const float* Q, const int8_t* K_int8, const int8_t* V_int8,
                      const float* k_scale, const float* v_scale, float* output, int num_heads,
                      int seq_len, int head_dim) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K_int8: cute.Tensor,
    V_int8: cute.Tensor,
    k_scale: cute.Tensor,
    v_scale: cute.Tensor,
    output: cute.Tensor,
    num_heads: cute.Int32,
    seq_len: cute.Int32,
    head_dim: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K_int8, V_int8, k_scale, v_scale are tensors on GPU
@jax.jit
def solve(
    Q: jax.Array,
    K_int8: jax.Array,
    V_int8: jax.Array,
    k_scale: jax.Array,
    v_scale: jax.Array,
    num_heads: int,
    seq_len: int,
    head_dim: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# Q, K_int8, V_int8, k_scale, v_scale, output are device pointers
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K_int8: UnsafePointer[Int8, MutExternalOrigin],
    V_int8: UnsafePointer[Int8, MutExternalOrigin],
    k_scale: UnsafePointer[Float32, MutExternalOrigin],
    v_scale: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    num_heads: Int32,
    seq_len: Int32,
    head_dim: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K_int8: torch.Tensor,
    V_int8: torch.Tensor,
    k_scale: torch.Tensor,
    v_scale: torch.Tensor,
    output: torch.Tensor,
    num_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K_int8: torch.Tensor,
    V_int8: torch.Tensor,
    k_scale: torch.Tensor,
    v_scale: torch.Tensor,
    output: torch.Tensor,
    num_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/96_int8_kv_cache_attention/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
